# Exploratory Data Analysis - NASA C-MAPSS FD001

Before building any model, it's important to understand the data first.
This notebook covers:
- Basic info and missing values check
- Engine lifetime distribution
- Sensor variance analysis
- Correlation between sensors and RUL
- Sensor trends over engine life
- Healthy vs degraded engine comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# column names - 2 id cols + 3 operational settings + 21 sensors
cols = ['unit_id', 'cycle', 'op1', 'op2', 'op3']
for i in range(1, 22):
    cols.append('sensor_' + str(i))

# load all 3 files
train_df = pd.read_csv('Data/train_FD001.txt', sep='\s+', header=None, names=cols)
test_df  = pd.read_csv('Data/test_FD001.txt',  sep='\s+', header=None, names=cols)
rul_df   = pd.read_csv('Data/RUL_FD001.txt',   sep='\s+', header=None, names=['RUL'])

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
print('RUL shape  :', rul_df.shape)

## 1. Missing Values Check

First thing to always check - are there any null values in the data?

In [ ]:
print('Missing values in train:', train_df.isnull().sum().sum())
print('Missing values in test :', test_df.isnull().sum().sum())
print()
print('No missing values - data is clean!')

## 2. Basic Data Info

In [ ]:
print('Data types:')
print(train_df.dtypes.value_counts())
print()
print('Number of engines in train:', train_df['unit_id'].nunique())
print('Number of engines in test :', test_df['unit_id'].nunique())

In [ ]:
# basic statistics of train data
train_df.describe().round(2)

## 3. Engine Lifetime Analysis

In training data, each engine runs until failure.
So the maximum cycle for an engine = its total lifetime.

In [ ]:
# max cycle per engine = how long it lasted
engine_cycles = train_df.groupby('unit_id')['cycle'].max()

plt.figure(figsize=(13, 4))
plt.bar(engine_cycles.index, engine_cycles.values, color='steelblue', edgecolor='black', linewidth=0.3)
plt.xlabel('Engine ID')
plt.ylabel('Total Cycles')
plt.title('How Long Each Engine Ran Before Failure')
plt.tight_layout()
plt.show()

print(f'Average lifetime : {engine_cycles.mean():.0f} cycles')
print(f'Shortest engine  : {engine_cycles.idxmin()} ({engine_cycles.min()} cycles)')
print(f'Longest engine   : {engine_cycles.idxmax()} ({engine_cycles.max()} cycles)')

## 4. Distribution of Engine Lifetimes

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(engine_cycles.values, bins=20, color='coral', edgecolor='black', alpha=0.8)
plt.axvline(engine_cycles.mean(), color='red', linestyle='--', label=f'Mean = {engine_cycles.mean():.0f}')
plt.xlabel('Total Cycles')
plt.ylabel('Number of Engines')
plt.title('Distribution of Engine Lifetimes')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Sensor Variance Analysis

Some sensors stay constant throughout - they have zero or near-zero variance.
These sensors give no information so we should drop them.

In [ ]:
sensor_cols = [c for c in cols if c.startswith('sensor_') or c.startswith('op')]
variances = train_df[sensor_cols].var().sort_values()

print('Sensor variances (sorted):\n')
for col, var in variances.items():
    flag = '  <-- drop (constant)' if var < 0.0001 else ''
    print(f'  {col:12s} : {var:.6f}{flag}')

# identify constant columns
zero_var_cols = variances[variances < 0.0001].index.tolist()
useful_sensors = [c for c in sensor_cols if c not in zero_var_cols]

print(f'\nColumns to drop : {zero_var_cols}')
print(f'Useful sensors  : {useful_sensors}')

## 6. Adding RUL Column

RUL = Remaining Useful Life = how many cycles left before failure.
Since each engine runs to failure: RUL = max_cycle - current_cycle

In [ ]:
max_cycles = train_df.groupby('unit_id')['cycle'].max().reset_index()
max_cycles.columns = ['unit_id', 'max_cycle']

train_eda = train_df.merge(max_cycles, on='unit_id')
train_eda['RUL'] = train_eda['max_cycle'] - train_eda['cycle']
train_eda.drop('max_cycle', axis=1, inplace=True)

print(f'RUL range: {train_eda["RUL"].min()} to {train_eda["RUL"].max()}')
train_eda[['unit_id', 'cycle', 'RUL']].head()

## 7. RUL Distribution

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(train_eda['RUL'], bins=50, color='mediumpurple', edgecolor='black', alpha=0.8)
plt.axvline(train_eda['RUL'].mean(), color='red', linestyle='--', label=f'Mean = {train_eda["RUL"].mean():.0f}')
plt.xlabel('RUL (Remaining Useful Life)')
plt.ylabel('Frequency')
plt.title('Distribution of RUL in Training Data')
plt.legend()
plt.tight_layout()
plt.show()

print('Right-skewed: most rows have low RUL because engines spend more time near failure.')

## 8. Correlation Heatmap - Sensors vs RUL

Which sensors are most related to RUL? A strong correlation means that sensor can help predict remaining life.

In [ ]:
corr_cols = useful_sensors + ['RUL']
corr_matrix = train_eda[corr_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap - Sensors vs RUL')
plt.tight_layout()
plt.show()

## 9. Top Sensors Correlated with RUL

In [ ]:
rul_corr = corr_matrix['RUL'].drop('RUL').abs().sort_values(ascending=False)

colors = ['green' if x > 0.5 else 'orange' if x > 0.3 else 'lightgray' for x in rul_corr.values]

plt.figure(figsize=(9, 5))
plt.barh(rul_corr.index, rul_corr.values, color=colors, edgecolor='black', linewidth=0.3)
plt.axvline(0.5, color='green', linestyle='--', alpha=0.6, label='Strong (> 0.5)')
plt.axvline(0.3, color='orange', linestyle='--', alpha=0.6, label='Moderate (> 0.3)')
plt.xlabel('Absolute Correlation with RUL')
plt.title('Sensor Correlation with RUL')
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 5 sensors correlated with RUL:')
for i, (sensor, val) in enumerate(rul_corr.head(5).items(), 1):
    print(f'  {i}. {sensor} = {val:.3f}')

## 10. Sensor Trends Over Engine Life

Plotting top 6 sensors for Engine 1 to see how they change as the engine degrades over time.

In [ ]:
engine1 = train_df[train_df['unit_id'] == 1]
top_sensors = rul_corr.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Sensor Readings Over Time - Engine 1', fontsize=13)

for idx, sensor in enumerate(top_sensors):
    ax = axes[idx // 3][idx % 3]
    ax.plot(engine1['cycle'], engine1[sensor], color='teal', linewidth=0.8)
    ax.set_title(sensor)
    ax.set_xlabel('Cycle')
    ax.set_ylabel('Value')

plt.tight_layout()
plt.show()

print('These sensors show a clear trend as the engine degrades - useful for prediction.')

## 11. Sensor Boxplots

Checking spread and outliers in each useful sensor.

In [ ]:
n = len(useful_sensors)
ncols = 5
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(17, 3 * nrows))
fig.suptitle('Boxplots of Useful Sensors', fontsize=13)

for idx, sensor in enumerate(useful_sensors):
    ax = axes[idx // ncols][idx % ncols]
    ax.boxplot(train_df[sensor], patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='navy'),
               medianprops=dict(color='red'))
    ax.set_title(sensor, fontsize=9)

# hide unused subplots
for idx in range(n, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

plt.tight_layout()
plt.show()

## 12. Healthy vs Degraded Comparison

Comparing top 6 sensors when engine is healthy (RUL > 125) vs near failure (RUL < 25).
If distributions look different, that sensor is helpful for prediction.

In [ ]:
healthy  = train_eda[train_eda['RUL'] > 125]
degraded = train_eda[train_eda['RUL'] < 25]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Sensor Values: Healthy (RUL > 125) vs Degraded (RUL < 25)', fontsize=13)

for idx, sensor in enumerate(top_sensors):
    ax = axes[idx // 3][idx % 3]
    ax.hist(healthy[sensor],  bins=30, alpha=0.6, label='Healthy',  color='green', density=True)
    ax.hist(degraded[sensor], bins=30, alpha=0.6, label='Degraded', color='red',   density=True)
    ax.set_title(sensor)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Clear distribution shift between healthy and degraded - these sensors are useful features.')

## EDA Summary

Key takeaways:

1. **No missing values** - dataset is clean
2. **100 engines** in both train and test sets
3. **Engine lifetimes** range from ~128 to ~362 cycles
4. **5 constant sensors** (op3, sensor_1, sensor_10, sensor_18, sensor_19) - will be dropped
5. **Top correlated sensors**: sensor_11, sensor_4, sensor_12, sensor_7 show strongest relation with RUL
6. **Clear degradation trends** visible in top sensors over engine lifetime
7. **Healthy vs degraded** distributions are clearly different for important sensors

These insights guide feature selection and help us understand the model later.